# DLMDSPWP01: Written Assignment End-to-End Pipeline
IU International University of Applied Sciences

This notebook demonstrates the complete stepwise workflow for the DLMDSPWP01 assignment:
1. Environment and Package Initialization
2. Data Ingestion and Validation (Pandas)
3. Least-Squares Regression and Ideal Function Selection (Direct Calculation and OOP)
4. Line-by-Line Test Data Classification (Stream Algorithm and OOP Generator)
5. Database Persistence via SQLAlchemy Core (SQLite Tables 1, 2, and 3)
6. Interactive Data Visualization (Bokeh Dashboard)
7. Automated Unit Testing (unittest Suite)
8. Version Control Workflow Documentation (Additional Task 1.3)

## Step 1: Environment and Package Initialization
Import required scientific libraries and reload the modular assignment package (Unit 1.5).

In [ ]:
import math
import os
import sys
import unittest
import pandas as pd
import sqlalchemy as db
from bokeh.plotting import figure, output_file, save
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, Band

# Ensure project root is available in module search path
if '..' not in sys.path:
    sys.path.insert(0, '..')
if '.' not in sys.path:
    sys.path.insert(0, '.')

import importlib
import assignment
import assignment.exceptions
import assignment.processors
import assignment.database
import assignment.visualizer

importlib.reload(assignment.exceptions)
importlib.reload(assignment.processors)
importlib.reload(assignment.database)
importlib.reload(assignment.visualizer)
importlib.reload(assignment)

from assignment import (
    AssignmentBaseException,
    DataLoadingError,
    DataValidationError,
    FunctionMatchingError,
    DatabaseOperationError,
    TrainingDatasetProcessor,
    IdealDatasetProcessor,
    TestDatasetProcessor,
    DatabaseManager,
    DataVisualizer,
)

print('Step 1 complete: Libraries imported and assignment package initialized.')

## Step 2: Exploratory Data Ingestion and Inspection
Load train.csv, ideal.csv, and test.csv to inspect dimensions, columns, and data types (Unit 3.2).

In [ ]:
train_path = '../data/train.csv' if os.path.exists('../data/train.csv') else 'data/train.csv'
ideal_path = '../data/ideal.csv' if os.path.exists('../data/ideal.csv') else 'data/ideal.csv'
test_path = '../data/test.csv' if os.path.exists('../data/test.csv') else 'data/test.csv'

train_df = pd.read_csv(train_path)
ideal_df = pd.read_csv(ideal_path)
test_df = pd.read_csv(test_path)

print(f'Training dataset: {train_df.shape[0]} rows, {train_df.shape[1]} columns')
print(f'Ideal functions dataset: {ideal_df.shape[0]} rows, {ideal_df.shape[1]} columns')
print(f'Test dataset: {test_df.shape[0]} rows, {test_df.shape[1]} columns')

print('\nTraining dataset preview (first 3 rows):')
print(train_df.head(3))

## Step 3: Mathematical Formulation and Ideal Function Selection
Your task is to write a Python-program that uses training data to choose the four ideal functions which are the best fit out of the fifty provided (C)*.

* The criterion is minimizing the sum of all squared y deviations (Least-Squares):

$$\text{SSE}_{k, j} = \sum_{i=1}^{N} (y_{\text{train}, k}^{(i)} - y_{\text{ideal}, j}^{(i)})^2$$

For each chosen pair, the maximum deviation is determined:

$$\text{max\_dev}_k = \max_{i=1}^{N} |y_{\text{train}, k}^{(i)} - y_{\text{ideal}, j^*}^{(i)}|$$

and the mapping threshold is derived as $\text{Threshold}_k = \text{max\_dev}_k \times \sqrt{2}$.

In [ ]:
# Step 3A: Direct Mathematical Calculation in the Notebook
training_cols = ['y1', 'y2', 'y3', 'y4']
ideal_cols = [f'y{i}' for i in range(1, 51)]

calculated_models = {}

for t_col in training_cols:
    sse_dict = {}
    max_dev_dict = {}
    for i_col in ideal_cols:
        deviation_series = train_df[t_col] - ideal_df[i_col]
        sse = (deviation_series ** 2).sum()
        max_d = deviation_series.abs().max()
        sse_dict[i_col] = sse
        max_dev_dict[i_col] = max_d

    best_ideal = min(sse_dict, key=sse_dict.get)
    max_dev = max_dev_dict[best_ideal]
    threshold = max_dev * math.sqrt(2)

    calculated_models[t_col] = {
        'chosen_ideal': best_ideal,
        'sse': float(sse_dict[best_ideal]),
        'max_dev': float(max_dev),
        'threshold': float(threshold)
    }

print('Step 3A (Direct Mathematical Calculation):')
print(f"{'Training Func':<15}{'Chosen Ideal':<15}{'Sum Squared Dev':<20}{'Max Dev':<12}{'Threshold (sqrt(2))':<15}")
for t_col, data in calculated_models.items():
    print(f"{t_col:<15}{data['chosen_ideal'].upper():<15}{data['sse']:<20.4f}{data['max_dev']:<12.4f}{data['threshold']:<15.4f}")

### Step 3B: Object-Oriented Encapsulation Verification
We now verify that the encapsulated TrainingDatasetProcessor.select_best_ideal_functions() method in assignment/processors.py produces identical mathematical results.

In [ ]:
train_processor = TrainingDatasetProcessor(train_path)
ideal_processor = IdealDatasetProcessor(ideal_path)

train_processor.load_data()
ideal_processor.load_data()

chosen_models = train_processor.select_best_ideal_functions(ideal_processor)

for t_col in training_cols:
    assert calculated_models[t_col]['chosen_ideal'].lower() == chosen_models[t_col]['chosen_ideal_col'].lower()
    assert math.isclose(calculated_models[t_col]['sse'], chosen_models[t_col]['sum_squared_deviation'])
    assert math.isclose(calculated_models[t_col]['max_dev'], chosen_models[t_col]['max_deviation'])

print('Step 3B: OOP TrainingDatasetProcessor method verified: Matches direct calculations.')

## Step 4: Line-by-Line Test Data Classification
Stream test points directly line-by-line from the CSV file using Python generator protocol (Unit 2.3).
Map each point if its deviation from the chosen ideal function does not exceed $\text{max\_dev} \times \sqrt{2}$.

In [ ]:
# Step 4A: Direct Line-by-Line Stream Classification
direct_mapped = []
direct_unmapped = []

ideal_indexed = ideal_df.set_index('x')
active_ideals = {t_col: data['chosen_ideal'] for t_col, data in calculated_models.items()}
thresholds = {data['chosen_ideal']: data['threshold'] for data in calculated_models.values()}

with open(test_path, 'r', encoding='utf-8') as f:
    header = f.readline()
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(',')
        tx = round(float(parts[0].strip()), 4)
        ty = float(parts[1].strip())

        best_ideal = None
        min_dev = float('inf')

        for t_col, i_col in active_ideals.items():
            iy = ideal_indexed.loc[tx, i_col]
            dev = abs(ty - iy)
            thresh = thresholds[i_col]
            if dev <= thresh:
                if dev < min_dev:
                    min_dev = dev
                    best_ideal = i_col.upper()

        if best_ideal is not None:
            direct_mapped.append({
                'X (test func)': tx,
                'Y (test func)': ty,
                'Delta Y (test func)': min_dev,
                'No. of ideal func': best_ideal
            })
        else:
            direct_unmapped.append((tx, ty))

print(f'Direct stream points evaluated: {len(direct_mapped) + len(direct_unmapped)}')
print(f'Direct stream points mapped: {len(direct_mapped)}')
print(f'Direct stream points unmapped: {len(direct_unmapped)}')

# Step 4B: Object-Oriented Verification via TestDatasetProcessor
test_processor = TestDatasetProcessor(test_path)
mapped_records, all_summary = test_processor.map_test_data(ideal_processor, chosen_models)

assert len(mapped_records) == len(direct_mapped)
print(f'OOP stream processing verified: {len(mapped_records)} records mapped.')

print('\nFirst 5 mapped records for Table 3:')
for record in mapped_records[:5]:
    print(record)

## Step 5: Database Persistence via SQLAlchemy Core
Compile SQLite database schema and populate Table 1 (training data), Table 2 (ideal functions), and Table 3 (mapped test data) per Section 1.2 specifications (Unit 3.5).

In [ ]:
db_output_path = '../outputs/assignment.db' if os.path.exists('../outputs') else 'outputs/assignment.db'
db_manager = DatabaseManager(db_output_path)

# 1. Compile schema
db_manager.compile_database_schema()

# 2. Persist datasets
db_manager.save_training_data(train_processor.get_dataframe())
db_manager.save_ideal_functions(ideal_processor.get_dataframe())
db_manager.save_test_data_mapping(mapped_records)

# 3. Verify record counts
count_t1 = db_manager.fetch_records_count('training_data')
count_t2 = db_manager.fetch_records_count('ideal_functions')
count_t3 = db_manager.fetch_records_count('test_data_mapping')

print(f'Table 1 (training_data) records: {count_t1}')
print(f'Table 2 (ideal_functions) records: {count_t2}')
print(f'Table 3 (test_data_mapping) records: {count_t3}')

## Step 6: Interactive Data Visualization with Bokeh
Generate an interactive HTML dashboard containing individual subplots for each chosen function pair with tolerance bands, training points, and assigned test cases with hover tooltips (Unit 3.4.3).

In [ ]:
html_output_path = '../outputs/visualization.html' if os.path.exists('../outputs') else 'outputs/visualization.html'
visualizer = DataVisualizer(html_output_path)

saved_html_path = visualizer.generate_visualization(
    train_processor.get_dataframe(),
    ideal_processor.get_dataframe(),
    chosen_models,
    all_summary
)

print(f'Interactive visualization saved: {saved_html_path}')
print(f'Dashboard size: {os.path.getsize(saved_html_path)} bytes')

## Step 7: Automated Unit Testing Verification
Execute all 19 automated test cases using Python standard unittest framework covering custom exceptions, loaders, math, generator streaming, and SQLite persistence (Unit 5.3).

In [ ]:
from tests.test_assignment import (
    TestCustomExceptions,
    TestDatasetLoaders,
    TestFunctionMatchingLogic,
    TestTestDataMapping,
    TestDatabaseManager,
    TestVisualizer,
    TestValidationEdgeCases,
)

suite = unittest.TestSuite()
test_classes = [
    TestCustomExceptions,
    TestDatasetLoaders,
    TestFunctionMatchingLogic,
    TestTestDataMapping,
    TestDatabaseManager,
    TestVisualizer,
    TestValidationEdgeCases,
]
for cls in test_classes:
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
test_results = runner.run(suite)

print(f'\nUnit test execution complete: {test_results.testsRun} tests run, '
      f'{len(test_results.failures)} failures, {len(test_results.errors)} errors.')

## Step 8: Version Control Workflow Documentation (Additional Task 1.3)
Git commands to clone the develop branch, implement changes on a feature branch, and open a pull request (Unit 6):

```bash
# 1. Clone develop branch locally
git clone -b develop https://github.com/ainaomotayo/DLMDSPWP01.git
cd DLMDSPWP01

# 2. Create feature branch
git checkout -b feature/model-matching

# 3. Stage changes
git add assignment/ main.py tests/ requirements.txt notebooks/ README.md

# 4. Commit changes
git commit -m "Implement ideal function matching pipeline and test suite"

# 5. Push feature branch to remote
git push origin feature/model-matching

# 6. Open Pull Request on GitHub targeting develop branch for peer review
```